# Practice 2: Weimar Jazz Database

---

Welcome to Practice 2.

In **Practice 1** you worked with **music21** on symbolic scores.

Today we do something different: we work with **tabular data** — rows and columns like a spreadsheet — and start applying the **statistical ideas from the lecture**. There is very little symbolic music processing in this notebook: most of the work is numbers and categories that describe tracks.

We will work with the [Weimar Jazz Database (WJazzD)](https://jazzomat.hfm-weimar.de/index.html) — a corpus of **456 transcribed jazz solos** from the history of jazz, from Louis Armstrong (1928) to contemporary postbop. We are not going to work with the melodies directly, but with the features extracted from the melodies. For each solo we have:

1. **A table containing features and metadata** (`Weimars_solos_with_features_and_metadata.csv`) — one row per solo, many numerical columns describing the melody (pitch range, interval entropy, event density, swing ratio, ...). These features were extracted with the **[MeloSpy](https://jazzomat.hfm-weimar.de/download/downloads/MSS_GUI_V_1_4_1.msi)** toolkit, which is specifically designed for jazz melody analysis. It was developed specifically to work with the Weimar Jazz Database and contains many more features than we use here. Each file also has metadata describing the performer, instrument, style, recording year, key, tempo, etc.
2. **The original MIDI files** — the symbolic transcriptions themselves, one `.mid` file per solo.

**What you will do in this notebook**

1. **Load** the features and metadata.
2. **Understand the features** — what each column means musically, and how features are grouped into families.
3. **Describe** the data with summary statistics and plots.
4. **Visualise features over time**, coloured by performer, to see how the corpus is structured.
5. **See which features are redundant** (highly correlated with each other).
6. Compare two famous players (**Parker vs. Davis**) with a **t-test**.
7. **Open two of their solos** as MIDI files with music21 and compare their features side-by-side.

In **Practice 3** we will continue with group-level analyses: one-way ANOVA across styles, correlation with recording year, and a chi-squared test of style vs. tonality.


## Part 1: Setup (Colab)

Run the cell below once at the start of the session. It installs the Python packages we need.


In [ ]:
# Install packages (Colab / first run). Safe to run again.
%pip install pandas numpy matplotlib seaborn scipy statsmodels music21 --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

RNG_SEED = 42
np.random.seed(RNG_SEED)

BG = "#F7F7F7"
# A palette for the six main jazz styles (plus FREE, which is rare)
STYLE_PALETTE = {
    "TRADITIONAL": "#8B7355",
    "SWING":       "#FFC000",
    "BEBOP":       "#C00000",
    "COOL":        "#AFCDCA",
    "HARDBOP":     "#7F3FBF",
    "POSTBOP":     "#3A3A3A",
    "FREE":        "#E07A5F",
}
plt.rcParams.update({
    "figure.facecolor": BG,
    "axes.facecolor": BG,
    "axes.edgecolor": "#888888",
    "axes.labelcolor": "#222222",
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "grid.color": "#CCCCCC",
    "grid.linestyle": "--",
    "grid.alpha": 0.5,
})

print("Libraries loaded. You're ready to go.")

---
## Part 2: Load the data

Our features are in a CSV file, which is a format storing tables of data. We use the **pandas** library to load the table into a Python-friendly structure called a **DataFrame**.

### What is pandas?

**[pandas](https://pandas.pydata.org/)** is a Python library for working with **tables** of data. The main object is a **DataFrame**: think of it as a spreadsheet where each **row** is one solo (one observation) and each **column** is one variable (e.g. tempo, genre).

In Practice 1 you saw **lists** and **loops**. Here we mostly call **ready-made functions** on a DataFrame: `df.head()`, `df.describe()`, `df.groupby(...)`, etc.

### Common DataFrame commands (you will see these a lot)

| Command | What it does |
|---------|----------------|
| `df.head()` | Shows the first 5 rows (a quick look at the table). |
| `df.info()` | Lists every column, its **dtype**, and how many **non-null** values there are. |
| `df.describe()` | Summary numbers for numeric columns: count, mean, standard deviation, min, max, quartiles. |
| `df.groupby("col")` | Splits the table by a category (e.g. genre) so you can compute means or counts **per group**. |
| `.round(3)` | Rounds numbers to 3 decimal places so tables are easier to read (not more "accurate"). |


In [ ]:
import urllib.request

url = "https://raw.githubusercontent.com/aljanaki/Digital_musicology/refs/heads/main/Practice%20sessions/Practice%202/Weimars_solos_with_features_and_metadata1.csv"
urllib.request.urlretrieve(url, "data.csv")

df = pd.read_csv("data.csv")

print("Table:", df.shape)

In [ ]:
df.head()

In [ ]:
df.info(verbose=True)

### Reading `df.info()`

- **Dtype**: The kind of value stored in the column. **`object`** usually means text; **`int64`** whole numbers; **`float64`** decimals; **`bool`** True/False.
- **Non-Null Count**: how many of the 632 rows actually have a value for this column. Features that weren't computable for some solos have fewer non-null values.


---
## Part 3: Understanding the features

The MeloSpy-extracted CSV has **256 columns**. That's a lot. Most of them fall into a handful of **feature families** that describe different aspects of the melody. It is worth spending a moment on what each family means — we won't use all of them, but recognising the naming convention makes it much easier to find what you want.

### Feature families (naming conventions)

| Prefix / pattern | What it describes | Example columns |
|---|---|---|
| `pitch_…` | Raw **MIDI pitch** statistics (min, max, mean, range, entropy, histogram) | `pitch_range`, `pitch_entropy`, `pitch_mean` |
| `pc_…` | **Pitch class** (pitch mod 12 — C, C#, D, …) distribution, ignoring octave | `pc_entropy`, `pc_hist_density_00_C` |
| `cpc_…` | **Chromatic pitch class** statistics, related to scale use | `cpc_entropy`, `cpc_zipf` |
| `tpc_…` | **Tonal pitch class** (key-relative) — same pitch class, but relative to the tonic of the piece | `tpc_entropy`, `tpc_hist_density_00` |
| `int_…` | Signed **intervals** in semitones between consecutive notes (positive = up) | `int_mean`, `int_max`, `int_entropy` |
| `abs_int_…` | **Absolute** intervals (ignore direction — how big is each jump?) | `abs_int_mean`, `abs_int_max` |
| `fuzzyint_…` | **Categorical** intervals — `step up`, `leap up`, `big jump up`, etc. | `fuzzyint_hist_01_big_jump_up` |
| `parsons_…` | **Parsons code** — just up / same / down (contour). Very coarse. | `parsons_hist_ascending` |
| `durclass_…` | Note **duration classes**: very short, short, medium, long, very long | `durclass_abs_hist_03_medium` |
| `ioi*_…` | **Inter-onset intervals** — time between note *starts* | `ioiclass_abs_mode`, `CV_ioi` |
| `event_density` | Notes **per second** (how fast the player is running) | `event_density` |
| `mean_swing_ratio`, `median_swing_ratio`, `std_swing_ratio` | **Swing**: beat/upbeat duration ratio (1.0 = straight, 2.0 = "textbook" swing) | `mean_swing_ratio` |
| `nPVI_dur`, `nPVI_ioi` | **Pairwise variability** of durations / IOIs — how uneven the rhythm is | `nPVI_ioi` |
| `syncopicity` | How syncopated the rhythm is | `syncopicity` |
| `avgtempo`, `mean_tempo` | Tempo in BPM | `avgtempo` |
| `ic_…` | **Interval class** (intervals mod 12, treating 7 and -5 the same) | `ic_entropy` |
| `aic_…`, `ric_…` | **Segment-length** features under different segmentation rules | `aic_mean_seg_len` |
| `adjacent_phrase_similarity_…` | How similar consecutive phrases are | `adjacent_phrase_similarity_std` |
| `ratio_chromatic_sequences`, `mean_length_chromatic_sequences` | Use of chromatic passing motion | `ratio_chromatic_sequences` |

### A short glossary of concepts

- **Entropy**: a measure of how "spread out" a distribution is. Higher entropy = more uniform use of all categories; lower entropy = a few categories dominate.
- **Zipf**: how well the frequency distribution follows a Zipf law (a power law common in natural sequences).
- **Circular statistics** (`*_circ_*`): used for pitch classes because they are arranged on a circle (C → C# → ... → B → C). Normal mean/std don't work on circular data.

### The core feature set we will use

The ones we'll focus on today:

| Feature | Meaning |
|---------|---------|
| `pitch_range` | Highest minus lowest note of the solo, in semitones. |
| `pitch_entropy` | How "varied" the pitch distribution is (higher = more pitch classes used roughly equally). |
| `int_mean` | Mean of signed semitone intervals between consecutive notes (positive = upward tendency). |
| `abs_int_mean` | Mean **absolute** interval size in semitones (ignores direction). |
| `int_entropy` | How varied the interval distribution is. |
| `event_density` | Notes per second — how "fast" the player is spitting out notes. |
| `avgtempo` | Tempo in BPM at which the solo was played. |
| `mean_swing_ratio` | Beat-upbeat ratio: 1.0 = straight eighths, 2.0 = "textbook" swing eighths. |
| `note_count` | Total number of notes in the solo. |

And from the metadata side: `performer`, `style`, `instrument`, `key`, `tonality_type`, `rhythmfeel`, `tempoclass`, `recordingyear`.


## Entropy

Let's pause for a new concept of entropy. Entropy measures how spread out or unpredictable a distribution is. If a solo uses only two or three pitches over and over, its pitch distribution is concentrated — entropy is low. If it uses all twelve pitch classes roughly equally, the distribution is flat — entropy is high. 

Imagine three solos. Solo A is one note repeated 100 times. Solo B uses two notes, 50 times each. Solo C uses all 12 pitch classes roughly equally. Which is the most "varied"? Obviously C. Which is the most "predictable"? A. Entropy puts a number on that intuition.

In [ ]:
# Quick look at the style distribution — this matters for hypothesis testing
print("Number of solos per style:")
print(df["style"].value_counts())

In [ ]:
# And the performers with the most solos
print("Top 10 performers by solo count:")
print(df["performer"].value_counts().head(10))

<div style="border: 2px solid #4A90D9; border-radius: 6px; padding: 12px; margin: 10px 0; background-color: #f0f7ff;">
<b>✏️ Task 1 — Explore the data</b><br><br>
In a new cell below, try <code>df.describe()</code> on just a few columns, e.g.:<br><code>df[['pitch_range', 'event_density', 'avgtempo']].describe()</code><br>Also try <code>df['instrument'].value_counts()</code> — which instruments dominate this corpus?<br><br>
<em>Bonus:</em> Pick one feature family from the table above (e.g. <code>fuzzyint_</code>) and list all columns that match it:<br><code>[c for c in df.columns if c.startswith('fuzzyint_')]</code>
</div>

---
## Part 4: Descriptive statistics and visualisation

Before we test anything, we **look** at the data. Summary statistics and plots are how we decide which tests make sense and which assumptions might be violated.


In [ ]:
# Summary statistics for a handful of musically interesting features
feat_cols = ["pitch_range", "pitch_entropy", "int_mean", "abs_int_mean",
             "event_density", "int_entropy", "avgtempo", "note_count"]
df[feat_cols].describe().round(2)

In [ ]:
# Histogram: event_density (notes per second) across all 456 solos
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df["event_density"].dropna(), bins=30, color="#9696E0", edgecolor="white")
ax.set_xlabel("Event density (notes per second)")
ax.set_ylabel("Count of solos")
ax.set_title("Distribution of event density across 456 jazz solos")
ax.grid(True)
plt.tight_layout()
plt.show()

print("Mean event_density:",   round(df["event_density"].mean(),   3))
print("Median event_density:", round(df["event_density"].median(), 3))

### What do we see?

Most solos cluster around **4–7 notes per second**, with a **long right tail** of very fast solos (10+ notes per second — think bebop heads taken at burnout tempo). Because of this tail, the **mean** is slightly higher than the **median**: the distribution is **right-skewed**.

This is typical of rate-like music features — they cannot go below zero, but there is no upper limit, so outliers always stretch the right side.


In [ ]:
# Boxplot: event_density by style
# Order styles historically (matches the chronology of jazz)
style_order = ["TRADITIONAL", "SWING", "BEBOP", "COOL", "HARDBOP", "POSTBOP"]

fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(
    data=df[df["style"].isin(style_order)],
    x="style", y="event_density", order=style_order,
    hue="style", hue_order=style_order, legend=False,
    palette=[STYLE_PALETTE[s] for s in style_order], ax=ax,
)
ax.set_title("Event density by jazz style (chronological order)")
ax.set_ylabel("Notes per second")
ax.grid(True, axis="y")
plt.tight_layout()
plt.show()

### How to read this boxplot

Each box shows the **interquartile range** (25th–75th percentile) for one style; the line inside is the **median**; the whiskers extend to typical extremes; dots are outliers.

Reading left-to-right is reading **jazz history**: Traditional (1920s) → Swing (30s–40s) → Bebop (mid-40s) → Cool (50s) → Hardbop (late 50s) → Postbop (60s–today).

The visual story is clear: **Bebop** and later styles tend to have **higher note density** than pre-war Traditional and Swing. This is a classic musicological claim — "bebop players play more notes" — and it's a great candidate for a statistical test later on.


<div style="border: 2px solid #4A90D9; border-radius: 6px; padding: 12px; margin: 10px 0; background-color: #f0f7ff;">
<b>✏️ Task 2 — Another feature</b><br><br>
Make a boxplot of <code>pitch_range</code> or <code>mean_swing_ratio</code> by style, using the same <code>style_order</code>. What story does each feature tell about the history of jazz?
</div>

---
## Part 5: How did features evolve over time?

A boxplot by **style** already tells a historical story, but style is a coarse label. Let's look at features **directly against recording year**, and colour each point by **performer**, so we can see who contributed which solos when.

We pick the top performers (by solo count) so the legend stays readable.


In [ ]:
# Pick the top N performers by number of solos so the plot is readable
TOP_N = 15
top_performers = df["performer"].value_counts().head(TOP_N).index.tolist()
print(f"Top {TOP_N} performers we'll highlight:")
for p in top_performers:
    years = df.loc[df["performer"] == p, "recordingyear"]
    print(f"  {p:25s}  n = {(df['performer']==p).sum():3d}   years: {int(years.min())}–{int(years.max())}")

In [ ]:
# A generic helper so we can reuse this plot for several features
def feature_over_time(feature, ax=None):
    """Scatter of `feature` vs recordingyear. Top performers are coloured;
    everyone else is shown in light grey as background."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(11, 5))

    bg = df[~df["performer"].isin(top_performers)][["recordingyear", feature]].dropna()
    ax.scatter(bg["recordingyear"], bg[feature],
               color="#BFBFBF", s=14, alpha=0.5, label="other performers")

    cmap = plt.get_cmap("tab10")
    for i, p in enumerate(top_performers):
        sub = df[df["performer"] == p][["recordingyear", feature]].dropna()
        ax.scatter(sub["recordingyear"], sub[feature],
                   color=cmap(i % 10), s=40, alpha=0.85,
                   edgecolor="white", linewidth=0.5, label=p)

    ax.set_xlabel("Recording year")
    ax.set_ylabel(feature)
    ax.set_title(f"{feature} over time — top {TOP_N} performers highlighted")
    ax.grid(True)
    ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=9, frameon=False)
    return ax

fig, ax = plt.subplots(figsize=(11, 5))
feature_over_time("event_density", ax=ax)
plt.tight_layout()
plt.show()

### Reading this plot

Each dot is one solo. The **x-axis is recording year**, the **y-axis is the feature value** (here, notes per second). Solos by the top eight performers are colour-coded; everyone else sits in the grey background.

A few things jump out:
- **Armstrong** (1920s–early 30s) sits in the bottom-left — early jazz, modest note density.
- **Parker** (mid-40s) has extraordinary event density for his era.
- **Coltrane** and **Rollins** in the late 50s / 60s explore a wider range of densities including some highest outliers ever.
- Individual performers tend to **cluster vertically** around their era.

This is also a reminder that the corpus is **not balanced** — some performers are overrepresented, some years have only one or two solos. Keep this in mind for any historical claim we make from this data.


In [ ]:
# Now do the same for pitch_entropy and mean_swing_ratio
fig, axes = plt.subplots(2, 1, figsize=(11, 10))
feature_over_time("pitch_entropy",    ax=axes[0])
feature_over_time("mean_swing_ratio", ax=axes[1])
# Only keep one legend, to save space
axes[0].get_legend().remove()
plt.tight_layout()
plt.show()

### What do these tell us?

- **Pitch entropy** (top) — there is an upward drift over the course of the century.
- **Mean swing ratio** (bottom) — there seems to be a downward drift (less swing) The early and swing-era players (Armstrong, Hawkins) cluster higher, around the "textbook" 2:1 swing. Later players (Parker, Davis, Coltrane) tend to sit lower, closer to straight eighths — bebop and modal players play with **less swing** in the mechanical sense, even when the music still feels like jazz.

These are the kinds of patterns we will formalise with **correlation** and **hypothesis tests** in the next practice session.


<div style="border: 2px solid #4A90D9; border-radius: 6px; padding: 12px; margin: 10px 0; background-color: #f0f7ff;">
<b>✏️ Task 3 — Your feature over time</b><br><br>
Call <code>feature_over_time("...")</code> with a feature of your choice — try <code>int_entropy</code>, <code>abs_int_mean</code>, <code>avgtempo</code>, or one of the chromatic-sequence features. Does the historical trend look strong, weak, or noisy?
</div>

---
## Part 6: Which features tell us the same thing?

The feature table has 256 columns — but **many of them are redundant**. If you measure a melody's pitch range, its pitch entropy, and its highest note, those three numbers are not independent. They move together.

Before picking features for a test, it's worth knowing **which features are highly correlated**, so you don't accidentally claim that "two different properties" both predict something — when really they are the same property wearing two hats.

A **correlation matrix** is the standard tool for this. Each cell is the Pearson correlation between two features (between −1 and +1). We visualise it as a heatmap.


In [ ]:
# A selection of musically interesting features. Grouped roughly by family.
selected = [
    # Pitch
    "pitch_range", "pitch_entropy", "pitch_std",
    # Interval
    "int_mean", "int_std", "abs_int_mean", "int_entropy", "int_range",
    # Rhythm / density
    "event_density", "note_count", "avgtempo",
    # Swing
    "mean_swing_ratio", "median_swing_ratio", "std_swing_ratio",
    # Structure
    "ratio_ascending_descending", "ratio_chromatic_sequences",
    "number_of_unique_pc",
]
# Keep only those actually in the dataframe
selected = [c for c in selected if c in df.columns]
corr = df[selected].corr()

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            annot=True, fmt=".2f", annot_kws={"size": 8},
            square=True, cbar_kws={"label": "Pearson r"}, ax=ax)
ax.set_title("Correlation matrix of selected features")
plt.tight_layout()
plt.show()

### What to look for

- **Red cells** (positive, close to +1) mean two features move together — if one is high, the other is high.
- **Blue cells** (negative, close to −1) mean they move in opposite directions.
- **Pale cells** near 0 mean the two features are essentially independent.

Some things you should see in this heatmap:

- `abs_int_mean` and `int_std` are **highly correlated** (~0.9+). That makes sense: if a solo has big leaps, both its average jump size and the spread of jumps will be large.
- `note_count` and `event_density` are moderately correlated — faster playing and longer solos both mean more notes, but they are not the same thing.
- The three swing-ratio features are tightly correlated with each other but nearly uncorrelated with pitch / interval features. Swing is its own axis.
- `pitch_range` and `pitch_entropy` are correlated but not perfectly — a solo can cover a wide range while repeating a few notes a lot (low entropy).

**Takeaway:** if you are building a multi-feature analysis, don't include both `abs_int_mean` and `int_std` as if they were independent evidence. They aren't.


In [ ]:
# Zoom into the most strongly correlated pairs
pairs = (corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
             .stack().dropna().sort_values(ascending=False))
print("Top 10 most strongly POSITIVELY correlated feature pairs:")
print(pairs.head(10).round(2))
print()
print("Top 5 most strongly NEGATIVELY correlated feature pairs:")
print(pairs.tail(5).round(2))

<div style="border: 2px solid #4A90D9; border-radius: 6px; padding: 12px; margin: 10px 0; background-color: #f0f7ff;">
<b>✏️ Task 4 — A scatter matrix</b><br><br>
Pick 4–5 features from the list above (e.g. <code>event_density</code>, <code>pitch_range</code>, <code>abs_int_mean</code>, <code>mean_swing_ratio</code>) and plot them with:<br>
<code>sns.pairplot(df[[...]].dropna(), plot_kws={'alpha': 0.4, 's': 10})</code><br>
Which pair looks <em>linearly</em> related? Which pair looks related but not linearly? Which looks totally independent?
</div>

---
## Part 7: Two-sample t-test — Parker vs. Davis

### Research question

> **Does Charlie Parker play more notes per second than Miles Davis?**

Charlie Parker (1920–1955) is the defining voice of **bebop**, famous for its breakneck tempos and dense lines. Miles Davis (1926–1991) played bebop early in his career but became a leading figure of **cool jazz** and later **modal jazz**, both of which favour more restrained, spacious phrasing. So musicologically we **expect** Parker to have higher `event_density`. Let's check it with a t-test.

- **H₀ (null):** Mean `event_density` is equal for Parker and Davis.
- **H₁ (alternative):** The means differ.

We will:
1. Check assumptions (**normality** with Shapiro–Wilk, **equal variance** with Levene).
2. Run Welch's t-test if variances differ, Student's t-test otherwise.
3. Report the **p-value** and **Cohen's d** (effect size).


In [ ]:
parker = df.loc[df["performer"] == "Charlie Parker",  "event_density"].dropna()
davis  = df.loc[df["performer"] == "Miles Davis",     "event_density"].dropna()

print(f"n Parker: {len(parker):3d}   mean = {parker.mean():.2f} notes/s")
print(f"n Davis:  {len(davis):3d}   mean = {davis.mean():.2f} notes/s")

In [ ]:
# Assumption checks
w_p, p_shap_p = stats.shapiro(parker)
w_d, p_shap_d = stats.shapiro(davis)
print(f"Shapiro–Wilk Parker: W = {w_p:.3f}, p = {p_shap_p:.3f}")
print(f"Shapiro–Wilk Davis:  W = {w_d:.3f}, p = {p_shap_d:.3f}")

w_lev, p_lev = stats.levene(parker, davis)
print(f"Levene (equal variance):  W = {w_lev:.3f}, p = {p_lev:.3f}")

equal_var = p_lev >= 0.05
print(f"Use equal variances in t-test? {equal_var}")

In [ ]:
# The t-test itself
t_stat, p_val = stats.ttest_ind(parker, davis, equal_var=equal_var)
print(f"t = {t_stat:.3f}")
print(f"p = {p_val:.4g}")

# Cohen's d with pooled standard deviation
def cohens_d(x, y):
    nx, ny = len(x), len(y)
    sx, sy = x.std(ddof=1), y.std(ddof=1)
    pooled = np.sqrt(((nx-1)*sx**2 + (ny-1)*sy**2) / (nx+ny-2))
    return (x.mean() - y.mean()) / pooled

d = cohens_d(parker, davis)
print(f"Cohen's d (Parker – Davis) = {d:+.2f}")
print("(|d| ≈ 0.2 small, 0.5 medium, 0.8 large)")

In [ ]:
# Visualize the two distributions
plot_data = pd.DataFrame({
    "event_density": pd.concat([parker, davis], ignore_index=True),
    "performer": ["Parker"]*len(parker) + ["Davis"]*len(davis),
})

fig, ax = plt.subplots(figsize=(7, 4))
sns.violinplot(
    data=plot_data, x="performer", y="event_density",
    hue="performer", legend=False,
    palette=["#C00000", "#3A3A3A"], ax=ax,
)
ax.set_title("Event density — Parker vs. Davis")
ax.set_ylabel("Notes per second")
ax.grid(True, axis="y")
plt.tight_layout()
plt.show()

What type of plot is this? What other types of plots are good for visualizing distributions like this?

### Interpretation

- **p-value**: If it's well below 0.05, we reject H₀ and conclude the two performers really do differ on average. A p-value of e.g. `0.0002` would mean: *if* Parker and Davis had identical mean event-density, we would see a difference this large only about 2 in 10 000 times by chance.
- **Cohen's d**: This tells us *how big* the difference is in standard-deviation units. For this dataset `d` typically comes out around **+1.5 or larger** — Parker really does play *substantially* more notes per second. That is a **very large** effect.

**Both matter.** A large d without a small p could mean your sample is too tiny; a small d with a tiny p could mean your difference is real but too small to be musically interesting. Always report and interpret both.


<div style="border: 2px solid #4A90D9; border-radius: 6px; padding: 12px; margin: 10px 0; background-color: #f0f7ff;">
<b>✏️ Task 5 — Another pairwise comparison</b><br><br>
Compare <b>John Coltrane</b> and <b>Sonny Rollins</b> on <code>event_density</code>. Run the full pipeline: Shapiro, Levene, t-test, Cohen's d.
</div>

---
## Part 8: Beyond event density — Parker vs. Davis on many features

The t-test gave us a confident answer on **one** variable: Parker plays faster. But the two musicians differ on many axes. Let's look at their average values across a handful of features side-by-side.


In [ ]:
compare_feats = [
    "event_density", "note_count", "pitch_range", "pitch_entropy",
    "int_mean", "abs_int_mean", "int_entropy",
    "mean_swing_ratio", "avgtempo",
]

summary = (df[df["performer"].isin(["Charlie Parker", "Miles Davis"])]
             .groupby("performer")[compare_feats]
             .agg(["mean", "std", "count"])
             .round(2))
summary

In [ ]:
# Same information as a grouped bar chart, z-scored so features with different
# scales can sit on the same y-axis
pair = df[df["performer"].isin(["Charlie Parker", "Miles Davis"])].copy()
z = pair[compare_feats].apply(lambda col: (col - df[col.name].mean()) / df[col.name].std())
z["performer"] = pair["performer"].values
means = z.groupby("performer").mean().T   # features on rows, performers on cols

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(means))
width = 0.38
ax.bar(x - width/2, means["Charlie Parker"], width, label="Charlie Parker", color="#C00000")
ax.bar(x + width/2, means["Miles Davis"],    width, label="Miles Davis",    color="#3A3A3A")
ax.axhline(0, color="#888", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(means.index, rotation=30, ha="right")
ax.set_ylabel("z-score vs. whole corpus")
ax.set_title("Parker vs. Davis across multiple features (standardised)")
ax.grid(True, axis="y")
ax.legend()
plt.tight_layout()
plt.show()

### What the bars tell us

Each bar shows how **far from the corpus average** Parker or Davis sits, in standard-deviation units. A bar at +1 means "one SD above the corpus average"; a bar at −1 means "one SD below".

Look for where the two players **point in opposite directions** (one bar up, one bar down). Those are the features that most sharply distinguish them. The ones where both bars point the same way tell you what they have in *common* (both are above-average in note count, for example — both play long solos).


<div style="border: 2px solid #4A90D9; border-radius: 6px; padding: 12px; margin: 10px 0; background-color: #f0f7ff;">
<b>✏️ Task 6 — Pick your own pair</b><br><br>
Swap Parker and Davis for a different pair (e.g. <b>Coltrane vs. Rollins</b>, or <b>Armstrong vs. Beiderbecke</b>) and regenerate this bar chart. Does the contrast you see in the plot match what you know about those two musicians?
</div>

---
## Part 9: From numbers back to music — loading the MIDIs with music21

Statistical features are powerful but they **abstract away** the music. Let's close the loop by going back to one of the solos we've been comparing and **looking at** it.

The WJazzD ships MIDIs in a zip archive. We'll download and unzip them, then pick a Parker solo and a Davis solo, inspect them with **music21**, and check that the features in the CSV match what we compute directly from the score.


In [ ]:
import urllib.request, zipfile, os

MIDI_URL = "https://jazzomat.hfm-weimar.de/download/downloads/RELEASE2.0_mid_unquant.zip"
ZIP_PATH = "wjazz_midis.zip"
MIDI_DIR = "wjazz_midis"

if not os.path.exists(MIDI_DIR):
    print("Downloading MIDI archive (~4 MB)...")
    urllib.request.urlretrieve(MIDI_URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(MIDI_DIR)
    print("Done.")
else:
    print("MIDI folder already exists — skipping download.")

# What's inside? Show a few
midi_files = []
for root, _, files in os.walk(MIDI_DIR):
    for f in files:
        if f.lower().endswith(".mid"):
            midi_files.append(os.path.join(root, f))
print(f"\n{len(midi_files)} MIDI files extracted.")
print("First five:")
for f in midi_files[:5]:
    print(" ", f)

In [ ]:
# Pick one Parker and one Davis solo to look at
parker_midis = [f for f in midi_files if "Parker" in f]
davis_midis  = [f for f in midi_files if "Davis"  in f or "MilesDavis" in f]

print(f"Parker MIDI files: {len(parker_midis)} total")
for f in parker_midis[:5]:
    print(" ", os.path.basename(f))

print(f"\nDavis MIDI files: {len(davis_midis)} total")
for f in davis_midis[:5]:
    print(" ", os.path.basename(f))

parker_path = parker_midis[0]
davis_path  = davis_midis[0]
print(f"\nWe'll work with:\n  {os.path.basename(parker_path)}\n  {os.path.basename(davis_path)}")

In [ ]:
from music21 import converter, note

def inspect_solo(midi_path):
    """Load a MIDI, compute a few features by hand, return a small dict."""
    score = converter.parse(midi_path)
    notes = [n for n in score.recurse().notes if isinstance(n, note.Note)]
    pitches = [n.pitch.midi for n in notes]
    return {
        "file": os.path.basename(midi_path),
        "n_notes (music21)": len(notes),
        "pitch_min (music21)": min(pitches),
        "pitch_max (music21)": max(pitches),
        "pitch_range (music21)": max(pitches) - min(pitches),
    }

parker_info = inspect_solo(parker_path)
davis_info  = inspect_solo(davis_path)
pd.DataFrame([parker_info, davis_info])

In [ ]:
# Now cross-check against the CSV
def find_row(midi_path):
    solo_id = os.path.basename(midi_path).replace(".mid", "") + ".sv"
    rows = df[df["id"] == solo_id]
    if len(rows) == 0:
        return None
    return rows.iloc[0]

parker_row = find_row(parker_path)
davis_row  = find_row(davis_path)

for label, row, info in [("Parker", parker_row, parker_info),
                         ("Davis",  davis_row,  davis_info)]:
    print(f"--- {label}: {info['file']} ---")
    if row is None:
        print("  (no matching row in features table)")
        continue
    print(f"  note_count    — music21: {info['n_notes (music21)']:4d}   MeloSpy: {int(row['note_count'])}")
    print(f"  pitch_range   — music21: {info['pitch_range (music21)']:4d}   MeloSpy: {int(row['pitch_range'])}")
    print(f"  event_density — MeloSpy: {row['event_density']:.2f} notes/s")
    print(f"  pitch_entropy — MeloSpy: {row['pitch_entropy']:.3f}")
    print()

### What just happened

We loaded a real MIDI transcription of a Parker and a Davis solo with music21 (same library as Practice 1), counted the notes, and computed the pitch range **ourselves**. Then we compared our hand-computed numbers with the values MeloSpy had pre-computed in the CSV.

You can see there are quite significant differences! These are related to how music21 handles grace notes and ornaments.

The following cell sets up musescore on colab computer. Do not run it on your windows! 

In [ ]:
## RUN THIS CELL WHEN ON COLAB. DO NOT RUN ON WINDOWS COMPUTERS

%%capture

# Install MuseScore – needed to display graphical notation
!add-apt-repository ppa:mscore-ubuntu/mscore-stable -y
!apt-get update
!apt-get install musescore

us = music21.environment.UserSettings()
us['musescoreDirectPNGPath'] = '/usr/bin/mscore'
us['directoryScratch'] = '/tmp'

Run the following lines if you are on Windows:

In [ ]:
import music21
us = music21.environment.UserSettings()

# Use a raw string (r"...") so you don't have to escape every backslash
us['musicxmlPath']           = r'C:\Program Files\MuseScore 4\bin\MuseScore4.exe'
us['musescoreDirectPNGPath'] = r'C:\Program Files\MuseScore 4\bin\MuseScore4.exe'

print(us['musescoreDirectPNGPath'])

In [ ]:
# Optional — display sheet music for one of the solos. In Colab this needs an
# external renderer (MuseScore / Lilypond) to show actual notation. If that is
# not configured, `.show('text')` is a safe fallback that prints the stream.
try:
    score = converter.parse(parker_path)
    # Uncomment the next line in an environment that has MuseScore / Lilypond installed
    #score.show()
    print("First 20 notes of the Parker solo (text view):")
    for n in [x for x in score.recurse().notes if isinstance(x, note.Note)][:20]:
        print(f"  {n.offset:6.2f}  {n.pitch.nameWithOctave:>5s}  dur={float(n.quarterLength):.2f}")
except Exception as e:
    print("Couldn't render notation in this environment:", e)

<div style="border: 2px solid #4A90D9; border-radius: 6px; padding: 12px; margin: 10px 0; background-color: #f0f7ff;">
<b>✏️ Task 7 — Your choice</b><br><br>
Pick a <b>different</b> MIDI file (a different Parker solo, or a different Davis solo, or a John Coltrane solo) and:<br>
1. Load it with <code>music21</code>.<br>
2. Display the sheet music (if your environment supports it) or the text view above.<br>
3. Count the notes and compute the pitch range by hand.<br>
4. Look up the same solo in <code>df</code> and check your numbers against MeloSpy's <code>note_count</code> and <code>pitch_range</code>.<br>
If they disagree, think about why — MeloSpy operates on the unquantized transcription, as do you, so the numbers should be extremely close.
</div>

---
## Summary

In this notebook we:
- Loaded the WJazzD feature table and understood the **families of features** MeloSpy extracts.
- Described the corpus with summary stats and boxplots.
- Looked at features **over time, coloured by performer**, to see that the corpus is uneven and that performers have characteristic "signatures".
- Built a **correlation heatmap** to see which features carry the same information (spoiler: `abs_int_mean` and `int_std` are nearly the same feature).
- Ran a **two-sample t-test** to confirm that Parker plays more notes per second than Davis, with a very large effect size.
- Compared Parker and Davis across **multiple features** at once, and opened their MIDI files in music21 to cross-check that MeloSpy's features match what we can compute ourselves.